# Paper Figures — Generated from Saved Results

It reads from `results/` CSVs and `client_data/` NPZs.
Run ALL experiment scripts first, then open this.

Figures produced:
- Fig 1: Non-IID RUL distributions (from preprocess.py)
- Fig 2: Health Index trajectories (from preprocess.py)
- Fig 3: Real vs. simulated signals
- Fig 4: Main results comparison table (bar chart)
- Fig 5: Ablation study bar chart
- Fig 6: RUL prediction curve + MC-Dropout confidence bands
- Fig 7: Convergence curves (loss per FL round)
- Fig 8: Statistical significance (Wilcoxon p-values)

In [ ]:
import os, sys
sys.path.insert(0, '..')

import numpy as np # pyright: ignore[reportMissingImports]
import pandas as pd # type: ignore
import matplotlib.pyplot as plt # type: ignore
import matplotlib.ticker as ticker # type: ignore
from scipy.stats import wilcoxon # type: ignore

RESULTS  = '../results'
FIG_DIR  = '../results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

FD_KEYS  = ['FD001', 'FD002', 'FD003', 'FD004']
PALETTE  = {'centralised': '#555555', 'fedavg': '#E74C3C',
             'fedprox': '#E67E22',    'proposed': '#2E86C1'}

plt.rcParams.update({'font.family': 'Arial', 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 150})
print('Setup done.')

## Fig 4: Main Results Comparison

In [ ]:
# Load all experiment CSVs and aggregate across seeds
def load_exp(name):
    path = os.path.join(RESULTS, f'{name}.csv')
    if not os.path.exists(path):
        print(f'  Missing: {path} — run the experiment first')
        return None
    return pd.read_csv(path)

exps = {name: load_exp(name) for name in ['centralised','fedavg','fedprox','proposed']}

# For each experiment and dataset: mean RMSE across seeds
summary = {}
for name, df in exps.items():
    if df is None: continue
    summary[name] = {
        fd: (df[f'{fd}_rmse'].mean(), df[f'{fd}_rmse'].std())
        for fd in FD_KEYS
    }

# Plot
if summary:
    x      = np.arange(len(FD_KEYS))
    n_exp  = len(summary)
    w      = 0.18
    offsets= np.linspace(-(n_exp-1)*w/2, (n_exp-1)*w/2, n_exp)

    fig, ax = plt.subplots(figsize=(12, 5))
    for (name, data), off in zip(summary.items(), offsets):
        means = [data[fd][0] for fd in FD_KEYS]
        stds  = [data[fd][1] for fd in FD_KEYS]
        ax.bar(x + off, means, w, yerr=stds,
               color=PALETTE.get(name,'#888'), label=name.capitalize(),
               capsize=3, alpha=0.9, edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(FD_KEYS, fontsize=12)
    ax.set_ylabel('RMSE (cycles)', fontsize=12)
    ax.set_title('RUL Prediction RMSE by Method and Client (mean ± std, 5 seeds)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, 'fig4_main_results.pdf')
    plt.savefig(path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved: {path}')

## Fig 5: Ablation Study

In [ ]:
abl_df = load_exp('ablation')

if abl_df is not None:
    # Average across seeds
    abl_summary = abl_df.groupby('variant').agg(
        overall_rmse_mean=('overall_rmse', 'mean'),
        overall_rmse_std =('overall_rmse', 'std'),
        overall_mae_mean =('overall_mae',  'mean'),
        description      =('description',  'first'),
    ).reset_index()

    # Order variants logically
    order = ['full','no_sim','no_sim_weight','no_phys_loss','no_attention']
    abl_summary['variant'] = pd.Categorical(abl_summary['variant'], order)
    abl_summary = abl_summary.sort_values('variant')

    fig, ax = plt.subplots(figsize=(11, 4.5))
    colors = ['#2E86C1','#E74C3C','#E67E22','#8E44AD','#1E8449']
    bars = ax.barh(
        abl_summary['description'],
        abl_summary['overall_rmse_mean'],
        xerr=abl_summary['overall_rmse_std'],
        color=colors, alpha=0.9, capsize=4, edgecolor='white'
    )
    # Annotate values
    for bar, val in zip(bars, abl_summary['overall_rmse_mean']):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=9)

    ax.set_xlabel('Overall RMSE (cycles)', fontsize=11)
    ax.set_title('Ablation Study — Impact of Each Component on RUL RMSE',
                 fontsize=12, fontweight='bold')
    ax.invert_yaxis()  # full system at top
    plt.tight_layout()
    path = os.path.join(FIG_DIR, 'fig5_ablation.pdf')
    plt.savefig(path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved: {path}')

## Fig 6: RUL Prediction Curves + MC-Dropout Confidence Bands

Run this cell AFTER loading a saved model checkpoint.

In [ ]:
import torch, yaml # type: ignore
from evaluate import make_loader, mc_predict
from models.tcn import build_model

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cpu')

# Load saved centralised model (or federated — swap path)
ckpt_path = os.path.join(RESULTS, 'centralised_seed42.pt')
if not os.path.exists(ckpt_path):
    print('Run train_centralised.py first to generate the checkpoint.')
else:
    model = build_model(cfg).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, fd in zip(axes, ['FD001', 'FD004']):
        npz = np.load(f'../client_data/{fd}.npz')
        loader = make_loader(npz['X_test'], npz['y_test'],
                             cfg['training']['batch_size'])
        mean_p, std_p, true_r = mc_predict(
            model, loader, device, n_samples=cfg['model']['mc_samples']
        )
        # Sort by true RUL for a clean plot
        idx = np.argsort(true_r)
        t   = true_r[idx]
        m   = mean_p[idx]
        s   = std_p[idx]

        ax.plot(t, t,    'k--', lw=1.2, alpha=0.5, label='Perfect prediction')
        ax.plot(t, m,    color='#2E86C1', lw=1.5,  label='Predicted RUL (mean)')
        ax.fill_between(t, m-2*s, m+2*s, alpha=0.25,
                        color='#2E86C1', label='95% MC-Dropout interval')
        ax.set_xlabel('True RUL (cycles)', fontsize=11)
        ax.set_ylabel('Predicted RUL (cycles)', fontsize=11)
        ax.set_title(f'{fd} — RUL Prediction with Uncertainty', fontweight='bold')
        ax.legend(fontsize=9)

    plt.suptitle('MC-Dropout Uncertainty-Aware RUL Prediction',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(FIG_DIR, 'fig6_rul_uncertainty.pdf')
    plt.savefig(path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved: {path}')

## Fig 7: Convergence Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, color in PALETTE.items():
    if name == 'centralised': continue
    path = os.path.join(RESULTS, f'{name}_convergence_seed42.csv')
    if not os.path.exists(path):
        print(f'  Missing: {path}')
        continue
    conv = pd.read_csv(path)
    if 'rmse' in conv.columns:
        ax.plot(conv['round'], conv['rmse'], color=color,
                lw=2, label=name.capitalize())

ax.set_xlabel('Communication Round', fontsize=11)
ax.set_ylabel('Global RMSE', fontsize=11)
ax.set_title('Federated Training Convergence', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
path = os.path.join(FIG_DIR, 'fig7_convergence.pdf')
plt.savefig(path, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved: {path}')

## Fig 8: Statistical Significance — Wilcoxon Signed-Rank Test

In [ ]:
proposed_df = load_exp('proposed')
baselines   = {name: load_exp(name) for name in ['centralised','fedavg','fedprox']}

if proposed_df is not None:
    print('=== Wilcoxon Signed-Rank Test (proposed vs. each baseline) ===')
    print('H0: No difference in overall_rmse  |  p < 0.05 → reject H0 → significant')
    print(f'{"Baseline":<15} {"Proposed RMSE":>16} {"Baseline RMSE":>16} {"p-value":>10} {"Sig.":>6}')
    print('-' * 68)

    proposed_rmse = proposed_df['overall_rmse'].values
    for name, df in baselines.items():
        if df is None: continue
        base_rmse = df['overall_rmse'].values
        # Align lengths (in case different seeds ran)
        n = min(len(proposed_rmse), len(base_rmse))
        try:
            stat, p = wilcoxon(proposed_rmse[:n], base_rmse[:n])
            sig = '**' if p < 0.01 else ('*' if p < 0.05 else 'ns')
            print(f'{name:<15} {proposed_rmse[:n].mean():>16.4f} '
                  f'{base_rmse[:n].mean():>16.4f} {p:>10.4f} {sig:>6}')
        except Exception as e:
            print(f'{name:<15}  Error: {e}')

    print()
    print('** p < 0.01,  * p < 0.05,  ns = not significant')
    print('Report these p-values in your paper Table 3.')